# YOLO11 训练 - HiLEx数据集 (Google Colab)

在Colab上使用免费GPU训练（T4 GPU，约15-20分钟）

In [ ]:
# 1. 安装依赖
!pip install ultralytics -q

In [ ]:
# 2. 克隆HiLEx数据集
!git clone https://github.com/HiLEx-DLA/HiLEx.git
%cd HiLEx/HiLex_Yolo_Format

In [ ]:
# 3. 修复data.yaml路径
import yaml

with open('data.yaml', 'r') as f:
    data = yaml.safe_load(f)

data['train'] = '/content/HiLEx/HiLex_Yolo_Format/train/images'
data['val'] = '/content/HiLEx/HiLex_Yolo_Format/valid/images'
data['test'] = '/content/HiLEx/HiLex_Yolo_Format/test/images'

with open('data_fixed.yaml', 'w') as f:
    yaml.dump(data, f)

!cat data_fixed.yaml

In [ ]:
# 4. 检查GPU
import torch
print('CUDA可用:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 5. 训练YOLO11n
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data='data_fixed.yaml',
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    patience=10,
    save=True,
    plots=True
)

print('训练完成！')

In [ ]:
# 6. 验证
model = YOLO('runs/detect/train/weights/best.pt')
metrics = model.val(data='data_fixed.yaml', split='test')

print(f'mAP@50: {metrics.box.map50:.3f}')
print(f'mAP@50-95: {metrics.box.map:.3f}')

In [ ]:
# 7. 导出OpenVINO格式
model.export(format='openvino', imgsz=640, half=True)

In [ ]:
# 8. 下载权重
from google.colab import files
import shutil

# 打包权重
!zip -r yolo11n_hilex.zip runs/detect/train/weights/

# 下载
files.download('yolo11n_hilex.zip')

print('下载完成！解压后使用best.pt或best_openvino_model/')